# 9장 실습: Vector + Graph 하이브리드 RAG

실습 가이드 9장 9-2 ~ 9-4의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | Recommendations (APOC 포함, 쓰기 권한 필요) |
| .env 설정 | Recommendations DB의 NEO4J 네 줄 (로컬 Docker라면 `bolt://localhost:7687`) |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

줄거리 임베딩이 아직 없는 DB라면 아래 1번 셀에서 한 번 만듭니다. 이미 있으면 그대로 넘어갑니다.

## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [1]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 연결하고 레이블별 노드 수를 출력한다. expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
# NEO4J는 접속 정보 묶음이다. 다른 LangChain 객체를 만들 때 **NEO4J로 그대로 넘긴다(실습 가이드 6-2).
# 연결 오류는 이 줄에서 난다. ServiceUnavailable: 서버가 꺼져 있거나 시작 중 / AuthError: 비밀번호 / APOC 오류: 플러그인 없음
graph, NEO4J = connect_graph(expect=["Movie", "User", "Genre"])

연결: bolt://localhost:7687 | DB: neo4j
{'label': 'Actor', 'n': 15443}
{'label': 'Movie', 'n': 9125}
{'label': 'Director', 'n': 3604}
{'label': 'User', 'n': 671}
{'label': 'Genre', 'n': 20}


In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_neo4j import Neo4jVector, GraphCypherQAChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

# 답변용 LLM과 임베딩 모델. 임베딩 모델은 줄거리를 임베딩할 때와 질문을 임베딩할 때 같아야 한다.
llm = ChatOpenAI(model="gpt-4o", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 줄거리 임베딩 (실습 가이드 9-2)

9장의 벡터 검색은 영화 줄거리를 벡터로 바꿔 둔 `Movie.embedding` 속성과 벡터 인덱스 `moviePlotIndex`가 있어야 동작합니다. 아래 셀의 `Neo4jVector.from_existing_graph`가 이 두 가지를 만듭니다. 내부에서는 다음 순서로 처리합니다.

1. `moviePlotIndex`가 없으면 만듭니다(1,536차원, cosine).
2. `embedding`이 없는 Movie를 1,000편씩 가져옵니다. 임베딩할 텍스트는 영화마다 `"\nplot:<줄거리>"` 형식입니다.
3. 1,000편의 텍스트를 OpenAI 임베딩 API에 한 번에 보내 벡터 1,000개를 받습니다.
4. `db.create.setNodeVectorProperty`로 각 Movie의 `embedding` 속성에 저장합니다.
5. 남은 영화가 없을 때까지 2~4를 반복합니다. 9,083편이면 10번입니다.

이미 임베딩된 영화는 건너뛰므로, 다시 실행해도 비용이 다시 들지 않고 중간에 끊겨도 이어서 할 수 있습니다.

| 항목 | 값 (2026-11-01 측정) |
|---|---|
| 임베딩 대상 | plot이 있는 영화 9,083편 (전체 9,125편) |
| 임베딩 텍스트 | 9,083개, 합계 315,376토큰 (편당 평균 35토큰) |
| 비용 | 1명 약 $0.006 (text-embedding-3-small, 100만 토큰당 $0.02 기준) |
| 시간 | 로컬 Docker에서 이 노트북 전체 실행 83초 |

학생이 직접 실행해도 되는가

비용과 시간은 문제가 되지 않습니다. 조건은 학생마다 쓰기 권한이 있는 Recommendations DB(APOC 포함)가 하나씩 있어야 한다는 것입니다.

| 운영 방식 | 방법 | 비고 |
|---|---|---|
| 학생이 직접 실행 | 아래 `RUN_EMBEDDING = True`로 한 번 실행 | 각자 DB가 필요합니다. Docker는 덤프 적재 1초, AuraDB는 Push to Aura에 수십 분이 걸립니다 |
| 임베딩된 DB를 나눠 줌 | 강사가 임베딩까지 마친 DB를 덤프 파일로 만들어 배포 | 학생은 Docker에 적재하거나 AuraDB 콘솔에서 가져오기만 하면 됩니다 |
| 강사 DB 하나를 같이 씀 | 강사 DB 접속 정보를 공유 | AuraDB Free는 계정이 하나라 학생도 쓰기 쿼리를 실행할 수 있어 권하지 않습니다 |

아래 셀은 먼저 Movie 노드가 있는지 확인합니다. 없으면 Recommendations가 아닌 DB에 연결된 것이므로 멈춥니다.

In [3]:
movie_count = graph.query("MATCH (m:Movie) RETURN count(m) AS n")[0]["n"]
# Movie가 없으면 이후 셀이 모두 엉뚱한 결과를 내므로 여기서 멈춘다(빈 DB나 POLE에 연결된 경우).
if movie_count == 0:
    raise RuntimeError("Movie 노드가 없습니다. .env가 Recommendations DB를 가리키는지 확인하고 커널을 재시작하세요.")

# True로 바꾸면 임베딩이 없는 영화만 임베딩한다 (처음 1회, 약 $0.006, 1~2분)
RUN_EMBEDDING = False

if RUN_EMBEDDING:
    # 7장과 같은 함수. 대상만 Document.text에서 Movie.plot으로 바뀌었다.
    Neo4jVector.from_existing_graph(
        embedding=embeddings,
        **NEO4J,
        index_name="moviePlotIndex",
        node_label="Movie",
        text_node_properties=["plot"],
        embedding_node_property="embedding",
    )

embedded = graph.query("MATCH (m:Movie) WHERE m.embedding IS NOT NULL RETURN count(m) AS n")[0]["n"]
print(f"Movie {movie_count}편 중 임베딩된 영화 {embedded}편")
if embedded == 0:
    print("임베딩이 없습니다. RUN_EMBEDDING = True로 바꿔 이 셀을 다시 실행하세요.")

Movie 9125편 중 임베딩된 영화 9083편


## 2. retrieval_query로 관계 정보 확장 (실습 가이드 9-3)

`retrieval_query`는 벡터 검색 결과인 `node`와 `score`를 받아 이어서 실행되는 Cypher입니다. `text`, `score`, `metadata` 세 항목을 반드시 반환해야 하고, 중간 `WITH`에서 `score`를 넘기지 않으면 오류가 납니다.

In [4]:
# retrieval_query: 벡터 검색이 찾은 노드(node)와 점수(score)를 받아 그래프를 더 탐색하는 Cypher.
# text, score, metadata 세 열을 돌려줘야 한다. text는 page_content, metadata는 Document.metadata가 된다.
# OPTIONAL MATCH를 쓰는 이유: 배우나 장르가 없는 영화도 검색 결과에서 빠지지 않게 하려고.
retrieval_query = """
WITH node, score
OPTIONAL MATCH (node)<-[:ACTED_IN]-(actor:Actor)
OPTIONAL MATCH (node)-[:IN_GENRE]->(genre:Genre)
WITH node, score, collect(DISTINCT actor.name) AS actors, collect(DISTINCT genre.name) AS genres
RETURN node.plot AS text,
       score,
       {title: node.title, actors: actors, genres: genres} AS metadata
"""

# from_existing_index: 임베딩은 이미 있으므로 인덱스에 연결만 한다(from_existing_graph와 달리 임베딩하지 않는다).
hybrid_store = Neo4jVector.from_existing_index(
    embedding=embeddings,
    index_name="moviePlotIndex",
    **NEO4J,
    retrieval_query=retrieval_query,
)
hybrid_retriever = hybrid_store.as_retriever(search_kwargs={"k": 5})

# 질문에 제목(Jumanji)이 없어도 줄거리의 뜻으로 찾고, 장르와 배우는 그래프에서 가져온다.
for d in hybrid_retriever.invoke("What is the movie about the magic jungle board game?"):
    print(d.metadata["title"], d.metadata["genres"], d.metadata["actors"][:3])

Jumanji ['Fantasy', 'Children', 'Adventure'] ['Bradley Pierce', 'Kirsten Dunst', 'Jonathan Hyde']
Rio 2 ['Comedy', 'Children', 'Animation', 'Adventure'] ['Jake T. Austin', 'Jemaine Clement', 'Kristin Chenoweth']
Oz the Great and Powerful ['IMAX', 'Fantasy', 'Adventure', 'Action'] ['Michelle Williams', 'Rachel Weisz', 'Mila Kunis']
Zathura ['Fantasy', 'Children', 'Adventure', 'Action'] ['Kristen Stewart', 'Dax Shepard', 'Josh Hutcherson']
Ferngully: The Last Rainforest ['Musical', 'Comedy', 'Children', 'Animation'] ['Jonathan Ward', 'Christian Slater', 'Samantha Mathis']


## 3. 벡터 문맥과 그래프 문맥을 합치는 LCEL 체인 (실습 가이드 9-4)

`RunnableParallel`이 벡터 검색과 Text2Cypher 그래프 조회를 함께 실행하고, 두 결과를 하나의 프롬프트에 넣습니다. 그래프 조회가 실패해도 벡터 문맥만으로 답하도록 예외를 처리합니다.

In [5]:
# 그래프 경로: 질문을 Cypher로 바꿔 실행한다(8장과 같은 체인, 프롬프트는 기본값).
movie_cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    validate_cypher=True,
    allow_dangerous_requests=True,
)

# 벡터 경로 결과(Document 리스트)를 LLM이 읽기 좋은 문자열로 바꾼다. 배우는 5명까지만 넣는다.
def format_movies(docs):
    return "\n\n".join(
        f"제목: {d.metadata['title']}\n장르: {', '.join(d.metadata['genres'])}\n"
        f"배우: {', '.join(d.metadata['actors'][:5])}\n줄거리: {d.page_content}"
        for d in docs
    )

# Cypher 생성이나 실행이 실패해도 체인 전체가 멈추지 않게 오류를 문자열로 돌려준다. 그러면 벡터 문맥만으로 답한다.
def graph_context(question: str) -> str:
    try:
        return movie_cypher_chain.invoke({"query": question})["result"]
    except Exception as e:
        return f"그래프 조회 실패: {e}"

# 두 문맥을 함께 주고, 충돌하면 그래프를 우선하라는 규칙을 둔다(그래프 결과는 DB에 있는 사실이므로).
fusion_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 영화 지식 도우미다. 벡터 검색 문맥과 그래프 조회 결과를 함께 고려해 사실에 근거한 한국어 답변을 작성한다. "
     "두 정보가 충돌하면 그래프 결과를 우선한다. 문맥에 없는 내용은 추측하지 않는다."),
    ("human",
     "질문: {question}\n\n[벡터 검색 문맥]\n{vector_context}\n\n[그래프 결과]\n{graph_context}\n\n"
     "요구사항: 가장 관련 있는 영화 1~3편의 제목, 장르, 주요 배우, 줄거리 요약을 제시하고 끝에 근거를 표기한다."),
])

# RunnableParallel: 같은 질문을 세 갈래로 동시에 보낸다.
#   question: 질문 그대로 / vector_context: 벡터 검색 + 포맷 / graph_context: Text2Cypher 결과
hybrid_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        vector_context=hybrid_retriever | format_movies,
        graph_context=RunnableLambda(graph_context),
    )
    | fusion_prompt
    | llm
    | StrOutputParser()
)

# 첫 질문은 줄거리 묘사(Toy Story), 둘째는 짧고 모호한 질문이다. 두 답변의 근거를 비교한다.
print(hybrid_chain.invoke("A cowboy doll and new spaceman figure in a boy's room?"))
print("=" * 40)
print(hybrid_chain.invoke("What is the movie about the board game?"))

1. **Toy Story**
   - **장르**: Fantasy, Comedy, Children, Animation, Adventure
   - **주요 배우**: Tom Hanks, Tim Allen, Jim Varney, Don Rickles
   - **줄거리 요약**: 카우보이 인형은 소년의 방에서 새로운 우주비행사 피규어가 자신을 대신해 최고 장난감 자리를 차지하자 깊은 위협을 느끼고 질투하게 된다.
   - **근거**: 벡터 검색 문맥 및 그래프 결과

2. **Explorers**
   - **장르**: Sci-Fi, Children, Adventure
   - **주요 배우**: Ethan Hawke, River Phoenix, Bobby Fite, Bradley Gregg
   - **줄거리 요약**: 50년대 외계인 영화에 집착하는 소년이 반복적으로 꿈에서 본 청사진을 친구와 함께 따라가며 우주선을 만들고 모험을 떠난다.
   - **근거**: 벡터 검색 문맥

3. **Iron Giant, The**
   - **장르**: Sci-Fi, Drama, Children, Animation, Adventure
   - **주요 배우**: Vin Diesel, Jennifer Aniston, Harry Connick Jr., James Gammon
   - **줄거리 요약**: 어린 소년이 외계에서 온 거대한 로봇과 친구가 되지만, 이를 파괴하려는 정부 요원과의 갈등을 겪는다.
   - **근거**: 벡터 검색 문맥
1. **Jumanji**
   - **장르**: Adventure, Family, Fantasy
   - **주요 배우**: Robin Williams, Kirsten Dunst, Bonnie Hunt
   - **줄거리 요약**: 두 아이가 오래된 보드 게임을 발견하고, 게임을 시작하면서 게임 속 세계가 현실로 나타나게 된다. 이들은 게임을 끝내기 위해 다양한 모험을 겪으며 게임의 규칙을 따라야 한다.
   - **근거**: 

## 실습 과제

위의 1~3번 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제, 정답 코드, 해설 순서로 되어 있습니다. 먼저 직접 작성해 본 뒤 정답 코드와 비교합니다.

| 과제 | 다루는 것 |
|---|---|
| 1 | `retrieval_query`에 관계 하나를 더 붙여 metadata 늘리기 |
| 2 | 벡터 검색 결과에서 4장의 협업 필터링(멀티홉)으로 확장하기 |
| 3 | 데이터에 없는 질문을 했을 때 하이브리드 RAG가 어떻게 답하는지 확인하고 고치기 |

### 과제 1. 감독 정보 추가

2번 셀의 `retrieval_query`를 고쳐, 검색된 영화의 감독 목록을 metadata의 `directors`로 함께 돌려주세요. 감독은 `(d:Director)-[:DIRECTED]->(node)` 관계로 연결되어 있습니다. "What is the movie about the magic jungle board game?"으로 검색해 영화 제목과 감독을 출력합니다.

In [6]:
# 과제 1 정답
# 기본 retrieval_query에서 metadata에 directors 한 줄만 추가했다.
# [(패턴) | 값]은 패턴 컴프리헨션: 영화마다 패턴에 맞는 값을 리스트로 모은다. 감독이 없으면 빈 리스트가 된다.
retrieval_query_dir = """
WITH node, score
OPTIONAL MATCH (node)<-[:ACTED_IN]-(actor:Actor)
OPTIONAL MATCH (node)-[:IN_GENRE]->(genre:Genre)
WITH node, score, collect(DISTINCT actor.name) AS actors, collect(DISTINCT genre.name) AS genres
RETURN node.plot AS text,
       score,
       {title: node.title, actors: actors, genres: genres,
        directors: [(node)<-[:DIRECTED]-(d:Director) | d.name]} AS metadata
"""

store_dir = Neo4jVector.from_existing_index(
    embedding=embeddings,
    index_name="moviePlotIndex",
    **NEO4J,
    retrieval_query=retrieval_query_dir,
)

for d in store_dir.as_retriever(search_kwargs={"k": 5}).invoke("What is the movie about the magic jungle board game?"):
    print(d.metadata["title"], "| 감독:", d.metadata["directors"])

Jumanji | 감독: ['Joe Johnston']
Rio 2 | 감독: ['Carlos Saldanha']
Oz the Great and Powerful | 감독: ['Sam Raimi']
Zathura | 감독: ['Jon Favreau']
Ferngully: The Last Rainforest | 감독: ['Bill Kroyer']


#### 과제 1 해설

실행 결과 (2026-11-01)

| 영화 | 감독 |
|---|---|
| Jumanji | Joe Johnston |
| Rio 2 | Carlos Saldanha |
| Oz the Great and Powerful | Sam Raimi |
| Zathura | Jon Favreau |
| Ferngully: The Last Rainforest | Bill Kroyer |

- `[(node)<-[:DIRECTED]-(d:Director) | d.name]`은 패턴 컴프리헨션입니다. 패턴에 맞는 경로마다 `|` 뒤의 값을 모아 리스트로 만듭니다. 감독이 없으면 빈 리스트가 됩니다.
- `OPTIONAL MATCH (node)<-[:DIRECTED]-(d:Director)`를 하나 더 붙여도 같은 결과가 나옵니다. 다만 배우 수 × 장르 수 × 감독 수만큼 행이 불어난 뒤 `collect(DISTINCT ...)`로 줄이게 되어 영화가 많을수록 느려집니다. 패턴 컴프리헨션은 이런 증폭 없이 목록을 바로 만듭니다.
- 감독을 답변에도 쓰려면 3번 셀의 `format_movies`에 `감독: {', '.join(d.metadata['directors'])}` 줄을 추가하고, 체인의 retriever를 `store_dir.as_retriever(...)`로 바꿉니다. metadata에 넣기만 하면 LLM에는 전달되지 않습니다.

### 과제 2. 함께 좋아한 영화 추가 (협업 필터링)

4장 추천 쿼리는 "이 영화를 평가한 사용자가 평가한 다른 영화"를 찾는 2-hop 쿼리였습니다. 이를 `retrieval_query`에 넣어, 벡터 검색으로 찾은 영화마다 다음 조건의 영화 5편을 metadata의 `alsoLiked`로 돌려주세요.

- 검색된 영화에 별점 4.0 이상을 준 사용자가
- 별점 4.0 이상을 준 다른 영화
- 그런 사용자가 많은 순서로 5편

"What is the movie about the magic jungle board game?"으로 상위 3편을 검색해 제목과 `alsoLiked`를 출력합니다.

In [7]:
# 과제 2 정답
# COLLECT { } 서브쿼리: 검색된 영화마다 따로 실행되어 결과를 리스트로 모은다. 안에서 ORDER BY, LIMIT을 쓸 수 있다.
# 4장 추천 쿼리와 같은 패턴이고, 두 영화 모두에 4.0 이상을 준 사용자만 센다.
retrieval_query_cf = """
WITH node, score
OPTIONAL MATCH (node)<-[:ACTED_IN]-(actor:Actor)
OPTIONAL MATCH (node)-[:IN_GENRE]->(genre:Genre)
WITH node, score, collect(DISTINCT actor.name) AS actors, collect(DISTINCT genre.name) AS genres
WITH node, score, actors, genres,
     COLLECT {
         MATCH (node)<-[r1:RATED]-(u:User)-[r2:RATED]->(rec:Movie)
         WHERE r1.rating >= 4.0 AND r2.rating >= 4.0
         WITH rec, count(u) AS users
         ORDER BY users DESC
         LIMIT 5
         RETURN rec.title
     } AS alsoLiked
RETURN node.plot AS text,
       score,
       {title: node.title, actors: actors, genres: genres, alsoLiked: alsoLiked} AS metadata
"""

store_cf = Neo4jVector.from_existing_index(
    embedding=embeddings,
    index_name="moviePlotIndex",
    **NEO4J,
    retrieval_query=retrieval_query_cf,
)

for d in store_cf.as_retriever(search_kwargs={"k": 3}).invoke("What is the movie about the magic jungle board game?"):
    print(d.metadata["title"], "| 함께 좋아한 영화:", d.metadata["alsoLiked"])

Jumanji | 함께 좋아한 영화: ['Forrest Gump', 'Jurassic Park', 'Aladdin', 'Terminator 2: Judgment Day', 'Shawshank Redemption, The']
Rio 2 | 함께 좋아한 영화: []
Oz the Great and Powerful | 함께 좋아한 영화: []


#### 과제 2 해설

실행 결과 (2026-11-01)

| 영화 | 함께 좋아한 영화 |
|---|---|
| Jumanji | Forrest Gump, Jurassic Park, Aladdin, Terminator 2: Judgment Day, Shawshank Redemption, The |
| Rio 2 | (없음) |
| Oz the Great and Powerful | (없음) |

- 벡터 검색(의미)으로 시작 영화를 찾고, 그래프 탐색(영화 → 사용자 → 다른 영화, 2-hop)으로 확장했습니다. 벡터만으로는 얻을 수 없는 "사람들의 평가 관계"가 답변 재료에 들어갑니다.
- `COLLECT { ... }`는 서브쿼리 결과를 리스트로 돌려줍니다. 조건에 맞는 사용자가 없는 Rio 2, Oz는 빈 리스트가 됩니다. `CALL { ... }`을 쓰면 결과가 없는 영화의 행 자체가 사라질 수 있습니다.
- `COLLECT { }`를 `RETURN`의 맵 안에 바로 쓰면 Neo4j 5.26에서 `Aggregation column contains implicit grouping expressions` 오류가 납니다. 정답처럼 앞의 `WITH` 단계에서 먼저 계산해 이름(`alsoLiked`)을 붙인 뒤 `RETURN`에서 씁니다.
- 결과가 Forrest Gump, Jurassic Park 같은 인기작으로 채워지는 것은 동시 평가 수로만 고르기 때문입니다(4장 과제 2와 같은 인기 편향).

### 과제 3. 데이터에 없는 영화를 물었을 때

3번 셀의 `fusion_prompt`에는 "문맥에 없는 내용은 추측하지 않는다"는 규칙이 있습니다. 이 규칙이 지켜지는지 확인합니다.

1. 데이터에 없을 법한 영화를 질문합니다. 예: "A cat becomes a space pirate and fights robot dolphins on Mars."
2. `hybrid_store.similarity_search_with_score()`로 벡터 검색 점수를 함께 보고, `hybrid_chain`의 답변을 확인합니다.
3. 답변이 비슷한 영화를 정답처럼 소개한다면, 어떻게 고칠 수 있을지 시도합니다.

In [8]:
# 과제 3 정답 (1) 점수와 기존 체인의 답변 확인
# 데이터에 없는 줄거리. 벡터 검색은 그래도 '가장 가까운' 영화를 항상 돌려준다.
question = "A cat becomes a space pirate and fights robot dolphins on Mars."

# 점수만으로 관련 여부를 가르기 어렵다는 것을 보인다. 앞의 Jumanji 질문 점수와 비교한다.
print("[벡터 검색 점수]")
for doc, score in hybrid_store.similarity_search_with_score(question, k=5):
    print(f"  {score:.3f}  {doc.metadata['title']}")

print("\n[기존 fusion_prompt 답변]")
print(hybrid_chain.invoke(question))

[벡터 검색 점수]
  0.752  Doom
  0.748  Red Planet
  0.739  Cat Returns, The (Neko no ongaeshi)
  0.728  Bolt
  0.725  Avatar

[기존 fusion_prompt 답변]
질문과 가장 관련 있는 영화는 다음과 같습니다:

1. **Red Planet**
   - **장르**: Thriller, Sci-Fi, Action
   - **주요 배우**: Benjamin Bratt, Tom Sizemore, Carrie-Anne Moss, Val Kilmer
   - **줄거리 요약**: 우주비행사들과 그들의 로봇 개 AMME가 죽어가는 지구를 구하기 위해 화성에서 해결책을 찾으려 하지만, 그들의 임무는 심각한 문제에 직면하게 됩니다.
   - **근거**: 화성을 배경으로 하며 로봇과 관련된 요소가 포함되어 있어 질문의 설정과 유사합니다.

2. **Doom**
   - **장르**: Sci-Fi, Horror, Action
   - **주요 배우**: Razaaq Adoti, Rosamund Pike, Deobia Oparei, Karl Urban
   - **줄거리 요약**: 우주 해병대가 화성의 연구 시설에서 이상한 사건을 조사하기 위해 파견되지만, 유전자 조작된 살인 기계들에 의해 위협을 받게 됩니다.
   - **근거**: 화성을 배경으로 하며, 전투와 관련된 요소가 있어 질문의 설정과 일부 유사합니다.

3. **Bolt**
   - **장르**: Comedy, Children, Animation, Adventure, Action
   - **주요 배우**: Mark Walton, Susie Essman, Miley Cyrus, John Travolta
   - **줄거리 요약**: 자신의 능력이 실제라고 믿는 공상 과학/액션 쇼의 개 스타가 그의 공동 출연자를 구하기 위해 전국을 횡단하는 여정을 떠납니다.
   - **근거**: 동물이 주인공으로 등장하며, 모험과 액션 

In [9]:
# 과제 3 정답 (2) LLM이 먼저 일치 여부를 판정하도록 프롬프트를 고친다
# fusion_prompt와 비교: 일치 여부 판정 단계, 일치하는 영화가 없을 때의 고정 답변, 지어내기 금지를 추가했다.
strict_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 영화 지식 도우미다. 벡터 검색 문맥과 그래프 조회 결과만 근거로 한국어로 답한다. "
     "먼저 각 후보 영화의 줄거리가 질문의 핵심 내용과 실제로 일치하는지 판단한다. "
     "일치하는 영화가 없으면 '데이터에서 질문과 일치하는 영화를 찾지 못했습니다.'라고만 답하고, 비슷한 영화를 정답처럼 소개하지 않는다. "
     "문맥에 없는 장르, 배우, 줄거리를 지어내지 않는다."),
    ("human",
     "질문: {question}\n\n[벡터 검색 문맥]\n{vector_context}\n\n[그래프 결과]\n{graph_context}\n\n"
     "요구사항: 일치하는 영화가 있으면 1~3편의 제목, 장르, 주요 배우, 줄거리 요약을 제시하고 끝에 근거를 표기한다."),
])

# 프롬프트만 바꾸고 나머지(벡터 경로, 그래프 경로)는 hybrid_chain과 같다.
strict_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        vector_context=hybrid_retriever | format_movies,
        graph_context=RunnableLambda(graph_context),
    )
    | strict_prompt
    | llm
    | StrOutputParser()
)

# 데이터에 있는 질문도 함께 넣어, 엄격한 프롬프트가 맞는 답까지 막지 않는지 확인한다.
for q in [question, "What is the movie about the magic jungle board game?"]:
    print("질문:", q)
    print(strict_chain.invoke(q))
    print("=" * 40)

질문: A cat becomes a space pirate and fights robot dolphins on Mars.
데이터에서 질문과 일치하는 영화를 찾지 못했습니다.
질문: What is the movie about the magic jungle board game?
제목: Jumanji  
장르: Fantasy, Children, Adventure  
배우: Bradley Pierce, Kirsten Dunst, Jonathan Hyde, Robin Williams  
줄거리: 두 아이가 마법의 보드 게임을 발견하고 플레이하면서, 수십 년 동안 그 안에 갇혀 있던 남자와 게임을 끝내야만 멈출 수 있는 여러 위험 요소들을 풀어놓게 된다.  
근거: 벡터 검색 문맥

제목: Zathura  
장르: Fantasy, Children, Adventure, Action  
배우: Kristen Stewart, Dax Shepard, Josh Hutcherson, Jonah Bobo  
줄거리: 두 어린 형제가 그들이 하고 있는 마법의 보드 게임에 의해 집이 우주 깊숙이 던져지면서 은하계 모험에 휘말리게 된다.  
근거: 벡터 검색 문맥


#### 과제 3 해설

실행 결과 (2026-11-01)

| 프롬프트 | "고양이 우주 해적" (데이터에 없음) | "magic jungle board game" (데이터에 있음) |
|---|---|---|
| 기존 `fusion_prompt` | Red Planet, Doom을 "질문과 가장 관련 있는 영화"로 소개 | Jumanji, Zathura |
| 판정 지시를 넣은 `strict_prompt` | "데이터에서 질문과 일치하는 영화를 찾지 못했습니다." | Jumanji, Zathura |

- 벡터 검색은 "없다"고 답하지 않습니다. 관련이 없어도 가장 비슷한 5편을 항상 돌려줍니다. 그래프 조회도 `CONTAINS "cat becomes a space pirate"` 같은 쿼리를 만들어 0행을 돌려줬습니다. 기존 프롬프트는 "가장 관련 있는 영화 1~3편을 제시하라"고 요구하므로, LLM은 화성·로봇이 나오는 영화를 골라 답했습니다. 지어낸 정보는 아니지만, 질문한 영화가 아닌 것을 답으로 내놓았습니다.
- 점수로 거르면 되지 않을까요? 같은 DB에서 질문별 1위 점수를 재 보면 그렇지 않습니다.

| 질문 | 데이터에 있나 | 1위 점수 |
|---|---|---|
| A cowboy doll and new spaceman figure in a boy's room? | 있음 (Toy Story) | 0.876 |
| What is the movie about the magic jungle board game? | 있음 (Jumanji) | 0.766 |
| A cat becomes a space pirate and fights robot dolphins on Mars. | 없음 | 0.752 (Doom) |
| What is the movie about the board game? | 있음. 그런데 Jumanji가 5위 안에 없음 | 0.715 |
| A Korean grandmother opens a kimchi restaurant on the Moon. | 없음 | 0.678 |

  데이터에 없는 질문(0.752)과 있는 질문(0.766)의 점수 차이가 0.014뿐이고, 데이터에 있는 질문(0.715)이 없는 질문보다 낮기도 합니다. 점수 기준값 하나로는 "관련 없음"을 가를 수 없습니다.
- 그래서 정답 (2)는 LLM에게 "후보의 줄거리가 질문과 실제로 일치하는지 먼저 판단하고, 없으면 찾지 못했다고 답하라"는 판정 단계를 맡겼습니다. 데이터에 없는 질문은 "찾지 못했습니다"로 답하고, 데이터에 있는 질문은 여전히 Jumanji, Zathura로 답합니다.
- 이 방법도 LLM의 판단이라 항상 맞지는 않습니다. 운영에서는 판정 결과와 근거를 함께 출력하게 하고, 질문 묶음으로 정답률을 시험해 프롬프트를 고칩니다.